# OCEAVERA — Environmental Data Collection

**Branch work notebook**

**Branch:** `pipeline/environmental-data-collection`  
**Review date:** 10 October 2026


## Contributors

Verified project identity for the work recorded in this branch notebook:

<table style="width:100%; border-collapse:separate; border-spacing:1rem .2rem; margin:.45rem 0 1.25rem;">
<thead>
<tr>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Registration Number</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Member Name</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">GitHub Account</th>
</tr>
</thead>
<tbody>

<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">IT24100184</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Ushan Srinuka</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><code>Ushan-Srinuka</code></td>
</tr>

</tbody>
</table>

This notebook records branch-level implementation and evidence. It is not an individual contribution statement or assessed reflection.


## 1. Purpose and scope

This notebook records the completed Bio-ORACLE v3 environmental-data acquisition and the source-validation handoff to the interim phase for the environmental data collection branch. The collector requested every regional catalogue grid and data variable in the agreed Sri Lanka EEZ-extrema rectangle.

### Included

- Catalogue inventory and all-variable NetCDF retrieval using the maintained Python collector.
- The rectangular extent **77.02333333333333–85.23291666666667° E** and **2.5665–11.44883333333333° N**.
- A request window of **2000-01-01 through 2100-01-01 UTC**, subject to coordinates actually published for each layer.
- Run and per-layer provenance receipts, plus an independent local integrity and request-scope audit.
- The D-038 source-validation handoff, which checks the completed run and receipts, copies publisher files unchanged to the stable interim path and records their provenance.

### Outside this branch

- Polygon clipping, a land or marine mask, or proof that returned cells are all marine.
- Scientific cleaning, ecological interpretation, layer compatibility decisions or predictor selection.
- Joining environmental layers to OBIS observations, feature extraction, modelling or evaluation.

Unrecorded biological observations are not confirmed absences. Future SSP layers are projections and do not extend the authorised assignment to future-climate species modelling.


## 2. Source data and provenance

The source is the Bio-ORACLE v3 catalogue exposed by its ERDDAP service. The collector preserves publisher-format NetCDF responses under `data/raw/bio_oracle/`, with one JSON receipt per layer and a catalogue run manifest. Raw payloads are ignored by Git.

The final run selected **356** regional grids from **357** catalogue rows and retrieved **2,392** variables. It completed with **4,017,532,960** payload bytes (about **3.742 GiB**). Its run identifier is `20261009T163625858012Z`. The current run report is selected from `data/raw/bio_oracle/_runs/` when this notebook executes.

The source-validation handoff is retained under `data/interim/bio_oracle/source_validation/`. Its aggregate manifest records the same run, 356 layers, 2,392 variables and 4,017,532,960 payload bytes. The notebook checks the staged file inventory and payload sizes, confirms the copied run manifest and compares staged receipts with their raw counterparts. It does not recalculate SHA-256 over the staged NetCDF payloads; those fingerprints remain recorded in the handoff manifest and were checked during the copy.

Bio-ORACLE provides decade-level coordinates, not annual observations. The downloaded baseline axes have labels 2000 and 2010; SSP axes have labels 2020 through 2090, and there is no separate 2100 coordinate. The 2090 SSP label is the final decade in the publisher's 2000–2100 horizon. One static terrain layer carries a publisher-labelled 1970 coordinate and is not time-filtered. Exact averaging endpoints for individual baseline slices remain unresolved.

The rectangle is an intake window formed from the minimum and maximum coordinates of the Sri Lankan EEZ feature. It is not the irregular EEZ polygon and does not define the eventual modelling domain.


## 3. Maintained artefacts and workflow

1. The [Bio-ORACLE collector](../../src/data_collection/bio_oracle_layers.py) inventories the live catalogue, queries the fixed rectangle and time window, retrieves every variable, and records run and layer manifests.
2. The [Bio-ORACLE staging script](../../src/data_preparation/stage_bio_oracle_layers.py) validates a complete run and receipt set, verifies each payload during copying, then publishes an unchanged copy under `data/interim/bio_oracle/source_validation/` with an aggregate provenance manifest. Raw files remain in place.
3. The [D-047 decision](../../docs/records/2026-10-09-decision-bio-oracle-catalog-intake.md) records catalogue-wide intake scope; [D-048](../../docs/records/2026-10-09-decision-bio-oracle-sri-lanka-bbox.md) records the selected rectangle; [D-038](../../docs/records/2026-10-08-decision-phase-specific-derived-data-paths.md) records the phase-specific path convention.
4. The [regional source record](../../docs/records/2026-10-09-source-bio-oracle-sri-lanka-catalog.md) preserves earlier deleted-run history and the final run's evidence.
5. The [data storage guide](../../docs/data/data-storage-and-provenance.md), [data index](../../data/README.md), [notebook conventions](../README.md) and [readiness review](../../docs/project/repository-readiness-and-alignment.md) describe retention, current status and downstream evidence gaps.

The notebook reads saved manifests, receipts and payloads only. It does not call the network or write dataset files.


## 4. Offline verification

Run the following Code cells in order from the repository root. They select the newest saved Bio-ORACLE run report, summarize its scope, stream each raw payload to verify its byte count, SHA-256, NetCDF signature and adjacent receipt, then reconcile the interim manifest with the source run. The handoff check validates staged file paths and payload sizes, compares each staged receipt byte-for-byte with its raw receipt, checks the copied run-manifest fingerprint, and audits the staged inventory. It does not reread and hash the staged NetCDF payloads. All data remain read-only; only small summaries are retained in notebook outputs. If the raw dataset has been removed, the notebook reports that no local run is available and makes no network request.

These checks establish transfer integrity and recorded query scope. They do not re-interpret the gridded scientific values or resolve the source's temporal aggregation and marine-mask limitations.


In [1]:
import hashlib
import json
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
ROOT = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "src" / "data_collection" / "bio_oracle_layers.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the OCEAVERA repository or one of its subfolders.")

RAW_ROOT = ROOT / "data" / "raw" / "bio_oracle"
RUN_REPORTS = sorted((RAW_ROOT / "_runs").glob("*_bio_oracle_catalog.json"))
if RUN_REPORTS:
    RUN_PATH = RUN_REPORTS[-1]
    RUN = json.loads(RUN_PATH.read_text(encoding="utf-8"))
    print("Project root: .")
    print("Python runtime:", sys.version.split()[0])
    print("Run report:", RUN_PATH.relative_to(ROOT).as_posix())
    print("No network request made; local files are read-only.")
else:
    RUN_PATH = None
    RUN = None
    print("No local Bio-ORACLE run report is available.")
    print("No network request made; no dataset files were written.")


Project root: .
Python runtime: 3.14.8
Run report: data/raw/bio_oracle/_runs/20261009T163625858012Z_bio_oracle_catalog.json
No network request made; local files are read-only.


In [2]:
if RUN is None:
    print("Run summary unavailable because no local run manifest was found.")
    print("Interim handoff audit unavailable because the raw run is missing.")
else:
    variable_count = sum(len(layer.get("variables", [])) for layer in RUN.get("layers", []))
    print("Run ID:", RUN["catalog_run_id"])
    print("Run status:", RUN["status"])
    print("Catalogue rows:", RUN["catalog_dataset_row_count"])
    complete_count = sum(
        layer.get("status") == "complete" for layer in RUN.get("layers", [])
    )
    reused_count = sum(
        layer.get("status") == "reused" for layer in RUN.get("layers", [])
    )
    print(
        "Regional layers complete/reused/planned:",
        f"{complete_count}/{reused_count}/{RUN['planned_layer_count']}",
    )
    print("Failed/pending layers:", RUN["failed_layer_count"], RUN["pending_layer_count"])
    print("Variables:", variable_count)
    print("Payload bytes:", sum(layer.get("response_bytes", 0) for layer in RUN.get("layers", [])))
    print("Longitude bounds:", RUN["requested_longitude_bounds_degrees_east"])
    print("Latitude bounds:", RUN["requested_latitude_bounds_degrees_north"])
    print("Requested temporal bounds:", RUN["requested_time_bounds_utc"])

    INTERIM_ROOT = ROOT / "data" / "interim" / "bio_oracle" / "source_validation"
    HANDOFF_PATH = INTERIM_ROOT / "manifest.json"
    handoff_errors = []
    if not HANDOFF_PATH.is_file():
        print("Interim handoff manifest: unavailable")
    else:
        HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
        handoff_layers = HANDOFF.get("layers", [])
        raw_layers = {layer.get("dataset_id"): layer for layer in RUN.get("layers", [])}
        expected_files = {
            HANDOFF_PATH.relative_to(ROOT).as_posix(),
            (INTERIM_ROOT / "catalog_run_manifest.json").relative_to(ROOT).as_posix(),
        }
        staged_payloads = 0
        staged_receipts = 0

        def resolve_staged_file(relative_path, label):
            candidate = (ROOT / relative_path).resolve()
            if not candidate.is_relative_to(INTERIM_ROOT.resolve()):
                handoff_errors.append((label, "path outside source-validation folder"))
                return None
            return candidate

        if HANDOFF.get("format") != "oceavera-bio-oracle-interim-source-validation-v1":
            handoff_errors.append(("manifest", "unexpected format"))
        if HANDOFF.get("source_run", {}).get("catalog_run_id") != RUN.get("catalog_run_id"):
            handoff_errors.append(("manifest", "source run ID mismatch"))
        if HANDOFF.get("layer_count") != len(handoff_layers) or len(handoff_layers) != len(raw_layers):
            handoff_errors.append(("manifest", "layer count mismatch"))

        for item in handoff_layers:
            dataset_id = item.get("dataset_id", "unknown")
            raw_layer = raw_layers.get(dataset_id)
            if raw_layer is None:
                handoff_errors.append((dataset_id, "layer absent from raw run"))
            elif (
                item.get("bytes") != raw_layer.get("response_bytes")
                or item.get("sha256") != raw_layer.get("sha256")
                or item.get("variables") != raw_layer.get("variables")
            ):
                handoff_errors.append((dataset_id, "handoff record disagrees with raw run"))

            payload_path = resolve_staged_file(item.get("staged_payload", ""), dataset_id + " payload")
            receipt_path = resolve_staged_file(item.get("staged_receipt", ""), dataset_id + " receipt")
            expected_files.update((item.get("staged_payload", ""), item.get("staged_receipt", "")))
            if payload_path is not None:
                if not payload_path.is_file():
                    handoff_errors.append((dataset_id, "missing staged payload"))
                elif payload_path.stat().st_size != item.get("bytes"):
                    handoff_errors.append((dataset_id, "staged payload size mismatch"))
                else:
                    staged_payloads += 1
            if receipt_path is not None:
                source_receipt_path = ROOT / item.get("source_receipt", "")
                if not receipt_path.is_file():
                    handoff_errors.append((dataset_id, "missing staged receipt"))
                elif not source_receipt_path.is_file():
                    handoff_errors.append((dataset_id, "missing source receipt"))
                elif receipt_path.read_bytes() != source_receipt_path.read_bytes():
                    handoff_errors.append((dataset_id, "staged receipt differs from source"))
                else:
                    staged_receipts += 1

        run_copy_reference = HANDOFF.get("source_run", {}).get("staged_copy", "")
        run_copy_path = resolve_staged_file(run_copy_reference, "run manifest")
        if run_copy_reference:
            expected_files.add(run_copy_reference)
        if run_copy_path is None or not run_copy_path.is_file():
            handoff_errors.append(("run manifest", "missing staged run manifest"))
        else:
            run_copy_bytes = run_copy_path.read_bytes()
            run_copy_hash = hashlib.sha256(run_copy_bytes).hexdigest()
            if len(run_copy_bytes) != HANDOFF.get("source_run", {}).get("bytes"):
                handoff_errors.append(("run manifest", "staged byte count mismatch"))
            if run_copy_hash != HANDOFF.get("source_run", {}).get("sha256"):
                handoff_errors.append(("run manifest", "staged checksum mismatch"))
            if RUN_PATH is not None and run_copy_bytes != RUN_PATH.read_bytes():
                handoff_errors.append(("run manifest", "staged copy differs from selected raw report"))

        inventory = [path for path in INTERIM_ROOT.rglob("*") if path.is_file()]
        actual_files = {path.relative_to(ROOT).as_posix() for path in inventory}
        inventory_bytes = sum(path.stat().st_size for path in inventory)
        if actual_files != expected_files:
            handoff_errors.append(("inventory", "file set differs from aggregate manifest"))
        if sum(item.get("bytes", 0) for item in handoff_layers) != HANDOFF.get("payload_bytes"):
            handoff_errors.append(("manifest", "payload byte total mismatch"))
        if sum(len(item.get("variables", [])) for item in handoff_layers) != HANDOFF.get("variable_count"):
            handoff_errors.append(("manifest", "variable count mismatch"))
        if HANDOFF.get("payload_bytes") != sum(layer.get("response_bytes", 0) for layer in RUN.get("layers", [])):
            handoff_errors.append(("manifest", "payload total differs from raw run"))

        print("Interim run ID:", HANDOFF.get("source_run", {}).get("catalog_run_id"))
        print("Interim layers and variables:", HANDOFF.get("layer_count"), HANDOFF.get("variable_count"))
        print("Staged payloads with expected sizes:", f"{staged_payloads}/{len(handoff_layers)}")
        print("Staged receipts byte-identical to raw:", f"{staged_receipts}/{len(handoff_layers)}")
        print("Interim files and inventory bytes:", f"{len(inventory):,}", f"{inventory_bytes:,}")
        print("Interim handoff and inventory audit:", "PASS" if not handoff_errors else f"FAIL ({len(handoff_errors)} issues)")
        print("Staged NetCDF SHA-256:", "recorded in handoff manifest; not recalculated by this notebook")
        for dataset_id, issue in handoff_errors[:10]:
            print("Handoff issue:", dataset_id, issue)


Run ID: 20261009T163625858012Z
Run status: complete
Catalogue rows: 357
Regional layers complete/reused/planned: 356/0/356
Failed/pending layers: 0 0
Variables: 2392
Payload bytes: 4017532960
Longitude bounds: [77.02333333333333, 85.23291666666667]
Latitude bounds: [2.5665, 11.44883333333333]
Requested temporal bounds: ['2000-01-01T00:00:00Z', '2100-01-01T00:00:00Z']
Interim run ID: 20261009T163625858012Z
Interim layers and variables: 356 2392
Staged payloads with expected sizes: 356/356
Staged receipts byte-identical to raw: 356/356
Interim files and inventory bytes: 714 4,021,176,513
Interim handoff and inventory audit: PASS
Staged NetCDF SHA-256: recorded in handoff manifest; not recalculated by this notebook


In [3]:
if RUN is None:
    print("Payload audit unavailable because no local run manifest was found.")
else:
    errors = []
    verified_payloads = 0
    verified_receipts = 0
    expected_lon = RUN["requested_longitude_bounds_degrees_east"]
    expected_lat = RUN["requested_latitude_bounds_degrees_north"]
    expected_time = RUN["requested_time_bounds_utc"]
    netcdf_signatures = (b"CDF", b"\x89HDF\r\n\x1a\n")

    for layer in RUN.get("layers", []):
        dataset_id = layer["dataset_id"]
        payload = ROOT / layer["payload_location"]
        receipt_path = ROOT / layer["manifest_location"]
        layer_status = layer.get("status")
        if layer_status not in {"complete", "reused"}:
            errors.append((dataset_id, "run status"))
        if not payload.is_file():
            errors.append((dataset_id, "missing payload"))
            continue

        digest = hashlib.sha256()
        byte_count = 0
        with payload.open("rb") as stream:
            signature = stream.read(8)
            digest.update(signature)
            byte_count += len(signature)
            while chunk := stream.read(8 * 1024 * 1024):
                digest.update(chunk)
                byte_count += len(chunk)
        checksum = digest.hexdigest()
        if byte_count != layer.get("response_bytes"):
            errors.append((dataset_id, "run byte count"))
        if checksum != layer.get("sha256"):
            errors.append((dataset_id, "run checksum"))
        if not any(signature.startswith(value) for value in netcdf_signatures):
            errors.append((dataset_id, "NetCDF signature"))
        verified_payloads += 1

        if not receipt_path.is_file():
            errors.append((dataset_id, "missing receipt"))
            continue
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        download = receipt.get("download", {})
        query = receipt.get("query", {})
        if receipt.get("status") != "complete" or receipt.get("dataset_id") != dataset_id:
            errors.append((dataset_id, "receipt identity/status"))
        receipt_run_id = receipt.get("catalog_run_id")
        if (
            layer_status == "complete"
            and receipt_run_id != RUN.get("catalog_run_id")
        ):
            errors.append((dataset_id, "receipt run ID"))
        elif layer_status == "reused" and (
            not isinstance(receipt_run_id, str)
            or not receipt_run_id
            or receipt_run_id == RUN.get("catalog_run_id")
        ):
            errors.append((dataset_id, "reused receipt run ID"))
        if download.get("response_bytes") != byte_count or download.get("sha256") != checksum:
            errors.append((dataset_id, "receipt fingerprint"))
        if query.get("variables") != layer.get("variables"):
            errors.append((dataset_id, "receipt variables"))
        if query.get("requested_longitude_bounds_degrees_east") != expected_lon:
            errors.append((dataset_id, "receipt longitude bounds"))
        if query.get("requested_latitude_bounds_degrees_north") != expected_lat:
            errors.append((dataset_id, "receipt latitude bounds"))
        if dataset_id == "terrain_characteristics":
            if query.get("requested_time_bounds_utc") is not None:
                errors.append((dataset_id, "static layer time filter"))
        elif query.get("requested_time_bounds_utc") != expected_time:
            errors.append((dataset_id, "receipt time bounds"))
        verified_receipts += 1

    partial_files = list(RAW_ROOT.rglob("*.part"))
    print(f"Payloads fingerprinted: {verified_payloads}/{len(RUN.get('layers', []))}")
    print(f"Layer receipts verified: {verified_receipts}/{len(RUN.get('layers', []))}")
    print("Partial response files:", len(partial_files))
    print("Integrity and scope audit:", "PASS" if not errors else f"FAIL ({len(errors)} issues)")
    if errors:
        for dataset_id, issue in errors[:10]:
            print("Issue:", dataset_id, issue)


Payloads fingerprinted: 356/356
Layer receipts verified: 356/356
Partial response files: 0
Integrity and scope audit: PASS


## 5. Result and limitations

The final collector run completed the full current catalogue plan for the selected rectangle: 356 layers and 2,392 variables. The offline Code cells independently verify the raw payloads and query receipts, then check that the source-validation manifest matches the selected run, that each staged payload has its recorded size, that staged receipts match the raw receipts, and that the interim inventory matches the manifest. The notebook does not recalculate the staged NetCDF payload hashes; the staging script records and verifies them during copy.

This source-validation handoff preserves publisher data and provenance. It does not perform scientific cleaning, spatial masking, occurrence matching, feature extraction, modelling or performance evaluation. Those stages remain subject to the project workflow, evidence gates and separate authorisation.


## 6. References

- [Bio-ORACLE catalogue](https://erddap.bio-oracle.org/erddap/tabledap/allDatasets.html)
- [Bio-ORACLE documentation](https://www.bio-oracle.org/documentation.php)
- [Bio-ORACLE collector](../../src/data_collection/bio_oracle_layers.py)
- [Bio-ORACLE raw-to-interim handoff](../../src/data_preparation/stage_bio_oracle_layers.py)
- [D-038: phase-specific data paths](../../docs/records/2026-10-08-decision-phase-specific-derived-data-paths.md)
- [D-047: catalogue-wide intake](../../docs/records/2026-10-09-decision-bio-oracle-catalog-intake.md)
- [D-048: Sri Lankan EEZ-extrema rectangle](../../docs/records/2026-10-09-decision-bio-oracle-sri-lanka-bbox.md)
- [Regional source record](../../docs/records/2026-10-09-source-bio-oracle-sri-lanka-catalog.md)
- [Data storage and provenance](../../docs/data/data-storage-and-provenance.md)
- [Current repository readiness](../../docs/project/repository-readiness-and-alignment.md)
